Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC SELECT * FROM (
# MAGIC WITH dminus2 AS (
# MAGIC   WITH sale_data AS (
# MAGIC     SELECT navi_customer_id, sale_date
# MAGIC     FROM corp_analytics.gi_jarvis_sale_master_base_tbl
# MAGIC   ),
# MAGIC   d_1_nonsales_conversations AS (
# MAGIC     SELECT * FROM (
# MAGIC       SELECT
# MAGIC           crm_ticket_id,
# MAGIC           agent_name AS Employee_Name,
# MAGIC           call_duration AS AHT,
# MAGIC           CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) AS call_Date,
# MAGIC           CAST(CAST(call_connect_ist AS TIMESTAMP) AS TIMESTAMP) AS call_Time,
# MAGIC           bt.navi_customer_id AS customer_id,
# MAGIC           sd.sale_date,
# MAGIC           bt.l0_disposition,
# MAGIC           CASE
# MAGIC             WHEN sd.sale_date = date_sub(current_date(), 2)
# MAGIC             THEN 'FALSE'
# MAGIC             ELSE 'd_1_nonsales_conversations'
# MAGIC           END AS filter_status,
# MAGIC           bt.insurance_type AS insurance_type
# MAGIC       FROM corp_analytics.gi_jarvis_datamodel_cte_tbl bt
# MAGIC       LEFT JOIN sale_data sd
# MAGIC         ON sd.navi_customer_id = bt.navi_customer_id
# MAGIC       WHERE CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) = date_sub(current_date(), 2)
# MAGIC         AND call_duration > 0
# MAGIC     ) t
# MAGIC     WHERE filter_status = 'd_1_nonsales_conversations'
# MAGIC   ),
# MAGIC   sale_data2 AS (
# MAGIC     SELECT
# MAGIC         navi_customer_id,
# MAGIC         sale_date,
# MAGIC         'd_1_sales_conversations' AS filter_status
# MAGIC     FROM corp_analytics.gi_jarvis_sale_master_base_tbl
# MAGIC     WHERE sale_date = date_sub(current_date(), 2)
# MAGIC   ),
# MAGIC   d_1_sales_customers_calls AS (
# MAGIC     SELECT
# MAGIC         crm_ticket_id,
# MAGIC         agent_name AS Employee_Name,
# MAGIC         call_duration AS AHT,
# MAGIC         CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) AS call_Date,
# MAGIC         CAST(CAST(call_connect_ist AS TIMESTAMP) AS TIMESTAMP) AS call_Time,
# MAGIC         bt.navi_customer_id AS customer_id,
# MAGIC         sd2.sale_date,
# MAGIC         bt.l0_disposition,
# MAGIC         sd2.filter_status,
# MAGIC         bt.insurance_type AS insurance_type
# MAGIC     FROM corp_analytics.gi_jarvis_datamodel_cte_tbl bt
# MAGIC     LEFT JOIN sale_data2 sd2
# MAGIC       ON sd2.navi_customer_id = bt.navi_customer_id
# MAGIC     WHERE sd2.filter_status = 'd_1_sales_conversations'
# MAGIC       AND CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) <= date_sub(current_date(), 2)
# MAGIC       AND call_duration > 0
# MAGIC   )
# MAGIC   SELECT * FROM d_1_nonsales_conversations
# MAGIC   UNION ALL
# MAGIC   SELECT * FROM d_1_sales_customers_calls
# MAGIC ),
# MAGIC
# MAGIC dminus1 AS (
# MAGIC   WITH sale_data AS (
# MAGIC     SELECT navi_customer_id, sale_date
# MAGIC     FROM corp_analytics.gi_jarvis_sale_master_base_tbl
# MAGIC   ),
# MAGIC   d_1_nonsales_conversations AS (
# MAGIC     SELECT * FROM (
# MAGIC       SELECT
# MAGIC           crm_ticket_id,
# MAGIC           agent_name AS Employee_Name,
# MAGIC           call_duration AS AHT,
# MAGIC           CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) AS call_Date,
# MAGIC           CAST(CAST(call_connect_ist AS TIMESTAMP) AS TIMESTAMP) AS call_Time,
# MAGIC           bt.navi_customer_id AS customer_id,
# MAGIC           sd.sale_date,
# MAGIC           bt.l0_disposition,
# MAGIC           CASE
# MAGIC             WHEN sd.sale_date = date_sub(current_date(), 1)
# MAGIC             THEN 'FALSE'
# MAGIC             ELSE 'd_1_nonsales_conversations'
# MAGIC           END AS filter_status,
# MAGIC           bt.insurance_type AS insurance_type
# MAGIC       FROM corp_analytics.gi_jarvis_datamodel_cte_tbl bt
# MAGIC       LEFT JOIN sale_data sd
# MAGIC         ON sd.navi_customer_id = bt.navi_customer_id
# MAGIC       WHERE CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) = date_sub(current_date(), 1)
# MAGIC         AND call_duration > 0
# MAGIC     ) t
# MAGIC     WHERE filter_status = 'd_1_nonsales_conversations'
# MAGIC   ),
# MAGIC   sale_data2 AS (
# MAGIC     SELECT
# MAGIC         navi_customer_id,
# MAGIC         sale_date,
# MAGIC         'd_1_sales_conversations' AS filter_status
# MAGIC     FROM corp_analytics.gi_jarvis_sale_master_base_tbl
# MAGIC     WHERE sale_date = date_sub(current_date(), 1)
# MAGIC   ),
# MAGIC   d_1_sales_customers_calls AS (
# MAGIC     SELECT
# MAGIC         crm_ticket_id,
# MAGIC         agent_name AS Employee_Name,
# MAGIC         call_duration AS AHT,
# MAGIC         CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) AS call_Date,
# MAGIC         CAST(CAST(call_connect_ist AS TIMESTAMP) AS TIMESTAMP) AS call_Time,
# MAGIC         bt.navi_customer_id AS customer_id,
# MAGIC         sd2.sale_date,
# MAGIC         bt.l0_disposition,
# MAGIC         sd2.filter_status,
# MAGIC         bt.insurance_type AS insurance_type
# MAGIC     FROM corp_analytics.gi_jarvis_datamodel_cte_tbl bt
# MAGIC     LEFT JOIN sale_data2 sd2
# MAGIC       ON sd2.navi_customer_id = bt.navi_customer_id
# MAGIC     WHERE sd2.filter_status = 'd_1_sales_conversations'
# MAGIC       AND CAST(CAST(call_connect_ist AS TIMESTAMP) AS DATE) <= date_sub(current_date(), 1)
# MAGIC       AND call_duration > 0
# MAGIC   )
# MAGIC   SELECT * FROM d_1_nonsales_conversations
# MAGIC   UNION ALL
# MAGIC   SELECT * FROM d_1_sales_customers_calls
# MAGIC )
# MAGIC
# MAGIC SELECT
# MAGIC     crm_ticket_id,
# MAGIC     Employee_Name,
# MAGIC     AHT,
# MAGIC     call_Date,
# MAGIC     call_Time,
# MAGIC     customer_id,
# MAGIC     sale_date,
# MAGIC     l0_disposition,
# MAGIC     filter_status,
# MAGIC     'dminus2' AS allocation_type,
# MAGIC     insurance_type
# MAGIC FROM dminus2
# MAGIC
# MAGIC UNION ALL
# MAGIC
# MAGIC SELECT
# MAGIC     crm_ticket_id,
# MAGIC     Employee_Name,
# MAGIC     AHT,
# MAGIC     call_Date,
# MAGIC     call_Time,
# MAGIC     customer_id,
# MAGIC     sale_date,
# MAGIC     l0_disposition,
# MAGIC     filter_status,
# MAGIC     'dminus1' AS allocation_type,
# MAGIC     insurance_type
# MAGIC FROM dminus1
# MAGIC ) final_data
# MAGIC
# MAGIC WHERE
# MAGIC     (dayofweek(current_date()) = 2 AND allocation_type IN ('dminus1', 'dminus2'))
# MAGIC     OR
# MAGIC     (dayofweek(current_date()) <> 2 AND allocation_type = 'dminus1')
# MAGIC
# MAGIC ORDER BY call_Date, call_Time, crm_ticket_id
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1yz_kDOtZzj5S7wJj_dVdZYFO7Us_SWxVf26-vtEoONs/edit?gid=0#gid=0"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=3
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=3
)